# Analisis de precios a partir de SEPA: procesamiento de datos
Basado en base [SEPA](https://datos.produccion.gob.ar/dataset/sepa-precios)

In [1]:
# =============================================================================
# CELDA 1: Setup y carga de canastables desde Drive
# =============================================================================
import pandas as pd
import numpy as np
import re
import unicodedata
from pathlib import Path

# Montar Google Drive (te va a pedir autorización — aceptá)
from google.colab import drive
drive.mount('/content/drive')

# Ruta a los parquets ya procesados
DRIVE_DIR = Path("/content/drive/MyDrive/sepa_analisis/2026-04-24")

# Chequeo: listar archivos disponibles
print(f"\n📂 Archivos en {DRIVE_DIR}:")
for f in sorted(DRIVE_DIR.glob("*.parquet")):
    peso_mb = f.stat().st_size / (1024**2)
    print(f"   {f.name:<30} {peso_mb:>7.2f} MB")

# Cargar canastables (lo único que necesitamos para 7B y 7C)
canastables = pd.read_parquet(DRIVE_DIR / "canastables.parquet")
print(f"\n✅ canastables cargado: {len(canastables):,} filas")
print(f"   Columnas: {list(canastables.columns)}")
print(f"\n📊 Distribución por rubro:")
print(canastables["rubro"].value_counts().to_string())

Mounted at /content/drive

📂 Archivos en /content/drive/MyDrive/sepa_analisis/2026-04-24:
   canastables.parquet               0.59 MB
   df_comercio.parquet               0.01 MB
   df_precios.parquet              219.27 MB
   df_sucursales.parquet             0.19 MB

✅ canastables cargado: 7,616 filas
   Columnas: ['id_producto', 'n_banderas', 'n_sucursales', 'n_provincias', 'precio_mediano', 'precio_p10', 'precio_p90', 'descripcion', 'marca', 'cant_presentacion', 'unidad_presentacion', 'ratio_p90_p10', 'desc_norm', 'rubro']

📊 Distribución por rubro:
rubro
Otros                  2146
Almacén                1071
Alcohol                 857
Higiene personal        675
Lácteos                 633
Panadería y harinas     530
Limpieza                495
Bebidas                 405
No relevante            237
Snacks y golosinas      227
Carnes y fiambres       226
Bebé                    114


In [2]:
# =============================================================================
# CELDA 2: Paso 7B — Deduplicación por concepto
# =============================================================================
# Objetivo: para cada "concepto de producto" (marca + tipo de producto) dejar
# un único representante, priorizando el de mayor cobertura.
# Ejemplo: los 4 formatos de fideos Lucchetti colapsan a 1 representante.
# =============================================================================

# -----------------------------------------------------------------------------
# Helpers (re-definidos porque arrancamos desde cero)
# -----------------------------------------------------------------------------
def normalizar_texto(s):
    if pd.isna(s):
        return ""
    s = str(s).lower()
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii")
    return s

# -----------------------------------------------------------------------------
# 7B.1) Definición de tokens a remover del "concepto"
# -----------------------------------------------------------------------------
# Cuando extraemos el concepto, eliminamos: unidades, empaques, conectores,
# tamaños y adjetivos genéricos. Lo que queda son las palabras que describen
# el TIPO de producto, no su formato ni presentación.
# =============================================================================

TOKENS_A_REMOVER = {
    # Unidades de medida
    "gr", "grs", "gra", "g", "kg", "kgs", "ml", "lt", "lts", "l", "cc",
    "cm3", "un", "uni", "ud", "uds", "unid", "unidad", "unidades",
    "mg", "mgs", "oz",
    # Empaques
    "bolsa", "bol", "botella", "bot", "lata", "latas", "sachet", "sach",
    "doypack", "doy", "pack", "paq", "paquete", "estuche", "est",
    "frasco", "fco", "caja", "cja", "pote", "pot", "tetra", "brick",
    "pouch", "sobre", "sob", "blister", "envase", "env",
    "x2", "x3", "x4", "x6", "x8", "x10", "x12", "x16", "x24",
    # Conectores
    "de", "del", "la", "el", "en", "con", "sin", "para", "por", "p",
    "d", "c", "s", "y", "o", "al", "a", "the",
    # Variantes que no distinguen concepto
    "clasico", "clasica", "clas", "original", "orig", "tradicional",
    "tradi", "promo", "promoc",
    "chico", "chica", "grande", "gde", "mediano", "med", "mini",
    "familiar", "fami", "famil", "individual", "indiv",
    "light", "lite", "diet", "zero", "cero",
    "nuevo", "nueva", "new", "x",
}


def extraer_concepto(desc_norm, marca, n_palabras=3):
    """
    Extrae las primeras N palabras significativas de la descripción,
    excluyendo tokens de formato/empaque/marca.

    Ejemplos:
      'manteca extra la serenisima' + marca 'LA SERENISIMA'
        → 'manteca extra'
      'fideos coditos lucchetti bolsa x 500 grs' + marca 'LUCCHETTI'
        → 'fideos coditos'
    """
    if pd.isna(desc_norm) or not desc_norm:
        return ""

    marca_norm = normalizar_texto(marca) if pd.notna(marca) else ""
    tokens_marca = set(marca_norm.split())

    # Tokenizar (sacar puntuación, dejar alfanuméricos)
    texto = re.sub(r"[^a-z0-9\s]", " ", desc_norm)
    palabras = texto.split()

    utiles = []
    for p in palabras:
        if re.fullmatch(r"\d+([.,]\d+)?", p):   # números puros
            continue
        if p in TOKENS_A_REMOVER:               # formato/empaque
            continue
        if p in tokens_marca:                   # parte de la marca
            continue
        if len(p) <= 2 and p not in {"te", "sal"}:  # muy cortas
            continue
        utiles.append(p)
        if len(utiles) >= n_palabras:
            break

    return " ".join(utiles)


# -----------------------------------------------------------------------------
# 7B.2) Filtrar a rubros válidos (sin 'Otros' ni 'No relevante')
# -----------------------------------------------------------------------------
RUBROS_VALIDOS = ["Almacén", "Alcohol", "Bebé", "Bebidas",
                   "Carnes y fiambres", "Higiene personal",
                   "Lácteos", "Limpieza", "Panadería y harinas",
                   "Snacks y golosinas"]

canasta_base = canastables[canastables["rubro"].isin(RUBROS_VALIDOS)].copy()
print(f"📋 Productos en rubros válidos: {len(canasta_base):,}")
print(f"   (descartamos 'Otros' y 'No relevante' para la canasta)")

# -----------------------------------------------------------------------------
# 7B.3) Extracción del concepto
# -----------------------------------------------------------------------------
print("\n🔧 Extrayendo concepto clave...")
canasta_base["concepto"] = [
    extraer_concepto(d, m)
    for d, m in zip(canasta_base["desc_norm"], canasta_base["marca"])
]

# Mostrar ejemplos de la extracción (control visual)
print("\n📋 Ejemplos de extracción (muestra aleatoria de 10):")
for _, r in canasta_base.sample(10, random_state=42).iterrows():
    print(f"   [{r['rubro'][:18]:<18}] {str(r['descripcion'])[:38]:<38} "
          f"({str(r['marca'])[:12]:<12}) → '{r['concepto']}'")

# -----------------------------------------------------------------------------
# 7B.4) Deduplicación
# -----------------------------------------------------------------------------
n_sin = (canasta_base["concepto"] == "").sum()
print(f"\n⚠️  Productos sin concepto extraíble (descartados): {n_sin}")
canasta_base = canasta_base[canasta_base["concepto"] != ""].copy()

n_antes = len(canasta_base)

# Ordenamos por cobertura descendente, después dedupliamos por
# (rubro, marca, concepto) manteniendo la primera ocurrencia → la de mayor cobertura
canasta_dedup = (canasta_base
    .sort_values(["n_banderas", "n_sucursales"], ascending=False)
    .drop_duplicates(subset=["rubro", "marca", "concepto"], keep="first")
    .reset_index(drop=True))

print(f"\n📉 DEDUPLICACIÓN:")
print(f"   Antes:    {n_antes:,} productos")
print(f"   Después:  {len(canasta_dedup):,} productos")
print(f"   Removidos: {n_antes - len(canasta_dedup):,} "
      f"({100*(n_antes-len(canasta_dedup))/n_antes:.1f}%)")

# -----------------------------------------------------------------------------
# 7B.5) QA: Top conceptos con más duplicación (ver que funcionó bien)
# -----------------------------------------------------------------------------
print("\n\n🔍 Top 10 conceptos que tenían más variantes:")
variantes = (canasta_base
    .groupby(["rubro", "marca", "concepto"])
    .size()
    .reset_index(name="n_variantes")
    .sort_values("n_variantes", ascending=False))
print(variantes.head(10).to_string(index=False))

# Mostrar detalle de los 3 casos top — para confirmar que la dedup
# eligió el mejor representante (marcado con ⭐)
print("\n\nDetalle de los 3 casos con más variantes:")
print("(⭐ = producto que quedó tras la deduplicación)")
ids_que_quedan = set(canasta_dedup["id_producto"].astype(str))
for _, row in variantes.head(3).iterrows():
    r, m, c, n = row["rubro"], row["marca"], row["concepto"], row["n_variantes"]
    print(f"\n--- {r} / {m} / '{c}' ({n} variantes) ---")
    subconj = canasta_base[
        (canasta_base["rubro"] == r) &
        (canasta_base["marca"] == m) &
        (canasta_base["concepto"] == c)
    ].sort_values("n_banderas", ascending=False)
    for _, v in subconj.iterrows():
        estrella = "⭐" if str(v["id_producto"]) in ids_que_quedan else "  "
        print(f"   {estrella} {str(v['descripcion'])[:45]:<45} "
              f"({v['cant_presentacion']:>6.0f} {str(v['unidad_presentacion']):<5}) "
              f"n_band={v['n_banderas']} med=${v['precio_mediano']:.0f}")

# -----------------------------------------------------------------------------
# 7B.6) Resumen por rubro post-deduplicación
# -----------------------------------------------------------------------------
print("\n\n📊 RESUMEN POR RUBRO (post-deduplicación):")
resumen = (canasta_dedup
    .groupby("rubro")
    .agg(n_productos=("id_producto", "count"),
         n_marcas=("marca", "nunique"),
         cobertura_media=("n_banderas", "mean"),
         cobertura_max=("n_banderas", "max"))
    .round(1)
    .sort_values("n_productos", ascending=False))
print(resumen.to_string())

# -----------------------------------------------------------------------------
# 7B.7) Persistir a Drive (no al cache local que se borra)
# -----------------------------------------------------------------------------
df_save = canasta_dedup.copy()
for col in df_save.select_dtypes(include=["category"]).columns:
    df_save[col] = df_save[col].astype(str)
df_save.to_parquet(DRIVE_DIR / "canasta_dedup.parquet", compression="snappy")

print(f"\n✅ Paso 7B completo.")
print(f"   Variable: canasta_dedup ({len(canasta_dedup):,} productos)")
print(f"   Guardado: {DRIVE_DIR / 'canasta_dedup.parquet'}")

📋 Productos en rubros válidos: 5,233
   (descartamos 'Otros' y 'No relevante' para la canasta)

🔧 Extrayendo concepto clave...

📋 Ejemplos de extracción (muestra aleatoria de 10):
   [Panadería y harina] ALIMENTO ARROZ S AZUCAR NESTUM X 225 G (NESTUM      ) → 'alimento arroz azucar'
   [Almacén           ] ESENCIA DE VAINILLA ALICANTE PET X 100 (ALICANTE    ) → 'esencia vainilla pet'
   [Carnes y fiambres ] ALIM PERR ADUL CARNE                   (PURIN       ) → 'alim perr adul'
   [Lácteos           ] YOGUR DESCREMADO VAINILLA GRAN COMPRA  (GRAN COMPRA ) → 'yogur descremado vainilla'
   [Lácteos           ] MANTECA S SAL TONADITA X 200 GRS       (TONADITA    ) → 'manteca sal'
   [Panadería y harina] FIDEOS FUSILLI CAJA BARILLA CAJA X 500 (BARILLA     ) → 'fideos fusilli'
   [Bebidas           ] AGUA S/GAS NARAN                       (LEVITE      ) → 'agua gas naran'
   [Limpieza          ] QUITAMANCHAS BOLILLA                   (TRENE       ) → 'quitamanchas bolilla'
   [Almacén      